# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

- - -
高级的推荐算法与评估技术！
首先，让我们深入探讨“摄取”这一概念：
1. 无需 LangChain！采用原生技术以实现最大灵活性2. 让我们利用语言模型来合理地划分这些部分吧
3. 让我们使用昨天所选用的最佳分块大小和编码器吧
4. 我们还应该让语言模型以最实用的方式（即“文档预处理”）对这些片段进行重新编写。

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

import os
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

api_key = os.getenv('EMBEDDING_API_KEY')
xjp_api_key = os.getenv('DASH_SCOPE_XJP_API_KEY')

load_dotenv(override=True)

MODEL = "gpt-4.1-nano"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

SELF_MODEL = "qwen3.5-plus-2026-02-15"
openai = OpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    # base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1",
    api_key=api_key,
)

In [3]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [3]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(
        description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query"
    )
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(
        description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,
                      metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

- - -
## 三个步骤：
1. 从知识库中获取文档，就像 LangChain 所做的那样
2. 调用语言模型将文档转换为分块形式
3. 将这些片段存储在色彩空间中
就是这样！
### Let's start with Step 1

In [4]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [5]:
documents = fetch_documents()

Loaded 76 documents


### Donezo! On to Step 2 - make the chunks

In [ ]:
split_system_prompt = """
### 系统提示词：Insurellm文档分类与切分指引
你是一名专业的文档分类专家，负责对Insurellm相关文档进行精准的类别切分与归类。请严格遵循以下规则完成文档分类，确保分类结果准确、依据清晰。

#### 一、核心分类体系
需将文档划分为以下4个核心类别，每个类别对应明确的内容范畴及参考示例：
|类别名称|核心内容范畴|
|---|---|
|company|公司整体相关信息，包括但不限于：公司发展历程、组织架构、企业文化、核心价值观、招聘/职业发展、公司概况/业务规模等|
|contract|公司与客户/合作方签订的正式合同、协议，包含合作条款、费用、权利义务、服务范围、签名等核心合同要素
|employees|员工个人相关的HR记录，包括但不限于：个人信息、职业晋升、绩效、薪酬、培训/认证、奖惩等|
|products|公司产品相关信息，包括但不限于：产品功能、定价、 roadmap、适用场景、核心价值、模块说明等|

#### 二、分类规则
1. **核心归属优先**：以文档的核心内容主题为分类依据，而非文件名（文件名仅作参考）；若文档内容跨多个类别，优先归为核心主题对应的类别，并标注次要关联类别。
2. **内容完整性判断**：需通读文档全文，基于文档整体内容判断类别，而非局部片段；例如仅提及产品名称的公司介绍文档，仍归为company类。
3. **无歧义归类**：若文档内容完全匹配某一类别的核心范畴，直接归为该类别，无需标注次要类别。

#### 三、输出格式要求
严格输出json格式
针对每一份待分类文档，输出以下内容：
```
1. 文档名称：[待分类文档名称]
2. 分类结果：[company/contract/employees/products]
3. 分类依据：[简要说明文档核心内容如何匹配对应类别的范畴，需结合上述分类体系的内容范畴描述]
4. 次要关联类别（如有）：[若有跨类别内容，标注次要类别；无则填“无”]
```

#### 四、执行要求
1. 严格遵守上述分类体系与规则，不得新增/合并类别，不得偏离核心内容范畴判断；
2. 分类依据需具体、可验证，避免模糊表述（例如：“文档核心为Alex Chen的HR记录，包含出生日期、薪酬、职业晋升等员工个人信息，符合employees类别的核心范畴”）；
3. 若文档内容无法匹配任何一类（极特殊情况），分类结果填“未归类”，并在分类依据中说明无法归类的原因。
"""

In [6]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

The output format is json.
Example:
Among them:
headline:A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query
summary:A few sentences summarizing the content of this chunk to answer common questions
original_text:The original text of this chunk from the provided document, exactly as is, not changed in any way
{{
"chunks":[
{{"headline":str,"summary":str,"original_text":str}},
]
}}
"""


In [7]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: knowledge-base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 5 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# About Insurellm

Insurellm was founded by Avery La

In [8]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [9]:
make_messages(documents[0])

[{'role': 'user',
  'content': '\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: company\nThe document has been retrieved from: knowledge-base/company/about.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don\'t leave anything out.\nThis document should probably be split into 5 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n# 

In [10]:
def process_document(document):
    schema = Chunks.model_json_schema()
    messages = make_messages(document)
    # response = completion(model=f"dashscope/{SELF_MODEL}",
    #                       messages=messages,
    #                       # response_format=Chunks,
    #                       response_format={"type": "json_object", "schema": schema},
    #                       api_key=api_key,
    #                       base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")
    completion = openai.chat.completions.create(
        model=SELF_MODEL,  # 您可以按需更换为其它深度思考模型
        messages=messages,
        response_format={"type": "json_object", "schema": schema},
        extra_body={"enable_thinking": False},
    )
    # reply = response.choices[0].message.content
    reply = completion.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]
    # return reply

In [14]:
change_doc = process_document(documents[0])

In [11]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [ ]:
chunks = create_chunks(documents)

  1%|▏         | 1/76 [00:19<24:26, 19.55s/it]

In [17]:
print(len(chunks))

644


### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [12]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [ ]:
create_embeddings(chunks)

In [13]:
from pymilvus import MilvusClient
from week5.embedding import text_embedding

v_collection_name = "vector_db_pro"
client = MilvusClient(
    uri="http://192.168.175.142:19530",
    db_name="default",
)

datas = []
for chunk in chunks:
    data = {
        "text": chunk.page_content,
        "doc_type": chunk.metadata["type"],
        "source": chunk.metadata["source"],
        "vector": text_embedding(chunk.page_content)
    }
    datas.append(data)



NameError: name 'chunks' is not defined

In [29]:
client.insert(
    collection_name=v_collection_name,
    data=datas
)

{'insert_count': 644, 'ids': [465630852002346916, 465630852002346917, 465630852002346918, 465630852002346919, 465630852002346920, 465630852002346921, 465630852002346922, 465630852002346923, 465630852002346924, 465630852002346925, 465630852002346926, 465630852002346927, 465630852002346928, 465630852002346929, 465630852002346930, 465630852002346931, 465630852002346932, 465630852002346933, 465630852002346934, 465630852002346935, 465630852002346936, 465630852002346937, 465630852002346938, 465630852002346939, 465630852002346940, 465630852002346941, 465630852002346942, 465630852002346943, 465630852002346944, 465630852002346945, 465630852002346946, 465630852002346947, 465630852002346948, 465630852002346949, 465630852002346950, 465630852002346951, 465630852002346952, 465630852002346953, 465630852002346954, 465630852002346955, 465630852002346956, 465630852002346957, 465630852002346958, 465630852002346959, 465630852002346960, 465630852002346961, 465630852002346962, 465630852002346963, 4656308520

# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [30]:

# chroma = PersistentClient(path=DB_NAME)
# collection = chroma.get_or_create_collection(collection_name)
# result = collection.get(include=['embeddings', 'documents', 'metadatas'])
# vectors = np.array(result['embeddings'])
# documents = result['documents']
# metadatas = result['metadatas']
# doc_types = [metadata['type'] for metadata in metadatas]
# colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in
#           doc_types]

In [40]:
from week5.milvus_op import get_all

all_data = get_all()
milvus_vectors = [v["vector"] for v in all_data]
np_vectors = np.array(milvus_vectors)

milvus_documents = [d["text"] for d in all_data]
milvus_doc_type = [dt["doc_type"] for dt in all_data]
milvus_colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in
                 milvus_doc_type]


In [44]:

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(np_vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=milvus_colors, opacity=0.8),
    # text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(milvus_doc_type, milvus_documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
                  scene=dict(xaxis_title='x', yaxis_title='y'),
                  width=800,
                  height=600,
                  margin=dict(r=20, b=10, l=10, t=40)
                  )

fig.show()

In [40]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(np_vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=milvus_colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(milvus_doc_type, milvus_documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [4]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [19]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
The output format is json.
for example:
{
order:[int]
}

"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    r_schema = RankOrder.model_json_schema()
    # response = completion(model=SELF_MODEL, messages=messages, response_format=RankOrder)
    response = openai.chat.completions.create(
        model=SELF_MODEL,  # 您可以按需更换为其它深度思考模型
        messages=messages,
        response_format={"type": "json_object", "schema": r_schema},
        extra_body={"enable_thinking": False},
    )
    reply = response.choices[0].message.content
    print(reply)
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [45]:
# RETRIEVAL_K = 10
#
#
# def fetch_context_unranked(question):
#     query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
#     results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
#     chunks = []
#     for result in zip(results["documents"][0], results["metadatas"][0]):
#         chunks.append(Result(page_content=result[0], metadata=result[1]))
#     return chunks

In [38]:
from week5 import embedding, milvus_op

RETRIEVAL_K = 10


def fetch_context_unranked(question):
    query_embedding = embedding.text_embedding(question)
    results = milvus_op.select_by_vector(query_embedding, 10)
    empty_chunks = []
    for result in results:
        m_d = {
            "doc_type": result["doc_type"],
            "source": result["source"]
        }
        text = result["text"]
        empty_chunks.append(Result(page_content=text, metadata=m_d))
    return empty_chunks


In [11]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [12]:
for chunk in chunks:
    print(chunk.page_content[:15] + "...")

HR Notes and De...
Annual Performa...
Data Engineer a...
Additional HR N...
Recent Performa...
Annual Performa...
Performance Rat...
Industry Recogn...
Annual Performa...
Awards Mentorsh...


In [21]:
reranked = rerank(question, chunks)

{
"order": [3, 1, 7, 2, 4, 5, 6, 8, 9, 10]
}
[3, 1, 7, 2, 4, 5, 6, 8, 9, 10]


In [22]:
for chunk in reranked:
    print(chunk.page_content[:15] + "...")

Data Engineer a...
HR Notes and De...
Performance Rat...
Annual Performa...
Additional HR N...
Recent Performa...
Annual Performa...
Industry Recogn...
Annual Performa...
Awards Mentorsh...


In [39]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    pc = c.page_content.lower()
    if "manchester" in pc:
        print(index)

In [32]:
reranked = rerank(question, chunks)

{
"order": []
}
[]


In [34]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [35]:
reranked[0].page_content

IndexError: list index out of range

In [36]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [37]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [ ]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
rewrite_query("Who won the IIOTY award?", [])

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [ ]:
answer_question("Who won the IIOTY award?", [])

In [ ]:
answer_question("Who went to Manchester University?", [])